# 04-01-08 Softmax-with-Loss 계층의 계산 그래프 유도

출처: 「Zero to 머신러닝 딥러닝 Master」 ([WikiDocs](https://wikidocs.net/book/21464)) — 이 노트북은 책의 코드 블록을 순서대로 담고 있습니다.


### Block 1 — 📖 처음부터 끝까지 친절하게 풀어보기 — 1) for문: 같은 작업을 여러 번 반복하기

In [ ]:
# ── 1) for문: 같은 작업을 여러 번 반복하기 ──
# range(5)는 0, 1, 2, 3, 4를 차례로 꺼내줍니다.
# 변수 이름이 필요 없을 때는 관례적으로 _ (밑줄)을 씁니다.
for _ in range(5):
    print("반복!")         # "반복!"이 5번 출력됩니다

# ── 2) *= 연산자: "기존 값에 곱해서 다시 저장" ──
# a *= 3  은  a = a * 3  과 똑같은 뜻입니다.
a = 2
a *= 3                      # a = 2 * 3 → a는 이제 6
print(a)                     # → 6

# ── 3) ** 연산자: 거듭제곱 ──
print(2 ** 10)                # → 1024  (2를 10번 곱한 것)
print(0.25 ** 3)              # → 0.015625  (0.25 × 0.25 × 0.25)

# ── 4) f-문자열: 변수 값을 문장 안에 바로 넣기 ──
# f"..." 안에서 {변수}를 쓰면, 그 자리에 변수 값이 들어갑니다.
이름 = "ResNet"
층수 = 152
print(f"{이름}은 {층수}층짜리 신경망입니다.")

# ── 5) :.2e 포맷: 아주 크거나 작은 숫자를 과학적 표기법으로 보기 ──
# 0.000000000000909를 일일이 쓰기 힘드니까, 9.09e-13처럼 줄여서 표시합니다.
작은수 = 0.25 ** 20
print(f"아주 작은 수: {작은수:.2e}")   # → 아주 작은 수: 9.09e-13

# ── 6) def: 나만의 함수 만들기 ──
# 자주 쓰는 계산을 묶어서 이름을 붙여 놓으면 편합니다.
def 인사하기(이름):
    return f"안녕하세요, {이름}님!"

print(인사하기("딥러닝"))     # → 안녕하세요, 딥러닝님!

### Block 2 — 04-01-08-A 깊이에는 두 얼굴이 있습니다 — 전화기 놀이를 떠올려 보세요 — 시그모이드 함수의 미분값은 아무리 커도 0.25입니다.

In [ ]:
# 시그모이드 함수의 미분값은 아무리 커도 0.25입니다.
# 층을 하나 지날 때마다 신호에 0.25가 곱해진다고 생각하면 됩니다.
시그모이드_미분_최대 = 0.25

# 처음 신호의 크기를 1.0(=100%)이라고 두고 시작합니다.
신호 = 1.0

# 층을 1개씩 지날 때마다 신호가 얼마나 남는지 확인해 봅시다.
for 층 in range(1, 21):        # 1층부터 20층까지
    신호 *= 시그모이드_미분_최대   # 매번 0.25를 곱합니다
    # 5층마다 한 번씩 중간 결과를 출력합니다
    if 층 % 5 == 0:
        print(f"{층}층 통과 후 남은 신호: {신호:.2e} (= {신호 * 100:.10f}%)")

### Block 3 — 04-01-08-C 덧셈 노드는 신호를 그대로 나눠주는 "만능 배달원" — 덧셈 y = a + b 의 역전파:

In [ ]:
# 덧셈 y = a + b 의 역전파:
# dL/da = dL/dy × 1 = dL/dy (그대로!)
# dL/db = dL/dy × 1 = dL/dy (그대로!)

# ── 순전파 ──
F_x = 0.001      # 합성곱 층들이 계산한 결과 (아주 작을 수 있음)
x   = 5.0        # 원래 입력 (지름길로 그대로 넘어옴)
y   = F_x + x     # ResNet의 핵심: y = F(x) + x
print(f"순전파: F(x)={F_x}, x={x}, y=F(x)+x={y}")

# ── 역전파 ──
dL_dy = 1.0       # 출력 쪽에서 전달받은 기울기

# 덧셈 노드의 역전파: 양쪽 모두에 기울기를 "그대로" 전달합니다.
dL_dFx = dL_dy * 1   # F(x) 쪽 기울기 (정식 코스)
dL_dx  = dL_dy * 1   # x 쪽 기울기 (지름길!) ← 이것이 핵심!

print(f"역전파: dL/dF(x)={dL_dFx}, dL/dx={dL_dx}")
print(f"→ 지름길(x) 쪽 기울기는 {dL_dx} 그대로! 절대 줄어들지 않습니다.")

### Block 4 — 04-01-08-D 코드로 지름길의 효과를 직접 확인해 봅시다

In [ ]:
import numpy as np

def sigmoid_grad_max():
    """시그모이드 함수의 미분 최댓값을 돌려줍니다.
    비유: '전화기 놀이에서 한 사람이 전달할 수 있는 최대 명확도'"""
    return 0.25

# 실험할 층 개수
층_개수 = 20

# ────────────────────────────────────────────
# [실험 1] 지름길이 "없는" 평범한 신경망
# ────────────────────────────────────────────
# 처음 신호 크기를 1.0(100%)으로 시작합니다.
누적_곱 = 1.0

# 매 층마다 0.25를 곱합니다 (전화기 놀이에서 말이 점점 작아지는 것과 같음).
for _ in range(층_개수):
    누적_곱 *= sigmoid_grad_max()

# :.2e는 과학적 표기법(예: 9.09e-13 = 0.000000000000909)
print(f"[실험1] 지름길 없음, {층_개수}층 후 남은 신호: {누적_곱:.2e}")

# ────────────────────────────────────────────
# [실험 2] 지름길이 "있는" ResNet 스타일
# ────────────────────────────────────────────
# 지름길이 있으면, 매 층마다 "변형된 값(0.25)"과 "원래 값(1)"이 더해집니다.
# 즉 (0.25 + 1) = 1.25가 매번 곱해지는 셈입니다.
누적_곱_resnet = 1.0

for _ in range(층_개수):
    누적_곱_resnet *= (sigmoid_grad_max() + 1)  # 0.25 + 1 = 1.25

print(f"[실험2] 지름길 있음, {층_개수}층 후 남은 신호: {누적_곱_resnet:.2e}")

# ────────────────────────────────────────────
# [비교] 두 결과를 나란히 비교
# ────────────────────────────────────────────
print(f"\n차이: 지름길이 있으면 신호가 {누적_곱_resnet / 누적_곱:.0f}배 더 강합니다!")

### Block 5 — 04-01-08-E VGGNet — "작은 레고 블록만으로 단순하게 쌓기" — 방법 A: 5×5 필터 한 번 쓰기

In [ ]:
# ── 방법 A: 5×5 필터 한 번 쓰기 ──
# 필터 크기가 5×5이므로, 파라미터 수 = 5 × 5 = 25개
파라미터_A = 5 * 5
print(f"방법 A (5×5 필터 1번): 파라미터 {파라미터_A}개")

# ── 방법 B: 3×3 필터 두 번 쓰기 (VGGNet 방식) ──
# 3×3 필터를 2번 쓰므로, 파라미터 수 = (3 × 3) + (3 × 3) = 18개
파라미터_B = (3 * 3) + (3 * 3)
print(f"방법 B (3×3 필터 2번): 파라미터 {파라미터_B}개")

# ── 비교 ──
절감률 = (1 - 파라미터_B / 파라미터_A) * 100
print(f"\n같은 영역(5×5)을 보지만, 방법 B가 파라미터 {절감률:.0f}% 절약!")
print(f"게다가 방법 B는 중간에 ReLU를 1번 더 거쳐서 표현력도 더 풍부합니다.")

# ── 수용 영역(Receptive Field) 확인 ──
# 3×3 필터를 처음 적용하면 3×3 영역을 봅니다.
# 그 위에 다시 3×3 필터를 적용하면, 원래 입력 기준으로 5×5 영역을 보게 됩니다.
# 공식: 수용 영역 = 필터크기 + (필터크기 - 1) × (층 수 - 1)
수용_영역_A = 5                           # 5×5 한 번 → 5×5
수용_영역_B = 3 + (3 - 1) * (2 - 1)     # 3×3 두 번 → 5×5
print(f"\n수용 영역 비교: A={수용_영역_A}×{수용_영역_A}, B={수용_영역_B}×{수용_영역_B} → 같음!")

### Block 6 — 04-01-08-F GoogLeNet — "세로 말고 가로로도 넓혀보자 (Inception 모듈)"

In [ ]:
import numpy as np

# ── Inception 모듈의 핵심 아이디어를 아주 단순하게 체험해 봅시다 ──
# 가짜 입력: 6×6 크기의 1채널 이미지
입력 = np.random.rand(6, 6)

# ── 경로 1: 1×1 필터 → "점 단위로 값을 변환" ──
# 1×1 합성곱은 각 픽셀에 하나의 숫자를 곱하는 것과 같습니다.
결과_1x1 = 입력 * 0.5   # 간소화: 모든 픽셀에 0.5를 곱함
print(f"경로1 (1×1): 출력 크기 = {결과_1x1.shape}")

# ── 경로 2: 3×3 필터 → "작은 무늬를 찾기" ──
# 3×3 영역을 슬라이딩하면, 출력 크기는 (6-3+1)=4 → 4×4
필터_3x3 = np.ones((3, 3)) / 9   # 평균 필터 (단순화)
결과_3x3 = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        영역 = 입력[i:i+3, j:j+3]               # 04-01-06 합성곱 복습
        결과_3x3[i, j] = np.sum(영역 * 필터_3x3)
print(f"경로2 (3×3): 출력 크기 = {결과_3x3.shape}")

# ── 경로 3: 5×5 필터 → "큰 무늬를 찾기" ──
# 5×5 영역을 슬라이딩하면, 출력 크기는 (6-5+1)=2 → 2×2
필터_5x5 = np.ones((5, 5)) / 25   # 평균 필터 (단순화)
결과_5x5 = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        영역 = 입력[i:i+5, j:j+5]
        결과_5x5[i, j] = np.sum(영역 * 필터_5x5)
print(f"경로3 (5×5): 출력 크기 = {결과_5x5.shape}")

# ── 핵심 관찰 ──
print(f"\n→ 세 경로가 서로 다른 크기의 출력을 만듭니다.")
print(f"→ 실제 Inception에서는 패딩으로 크기를 맞춘 후 '나란히 이어붙이기'합니다.")
print(f"→ 이것이 '분식 모듬 세트'처럼 다양한 정보를 한꺼번에 모으는 비결입니다!")

### Block 7 — 04-01-08-EF VGGNet vs GoogLeNet vs ResNet — 한눈에 비교 — 유명 구조들의 파라미터 수와 층 수를 비교합니다

In [ ]:
# 유명 구조들의 파라미터 수와 층 수를 비교합니다
구조들 = {
    "VGGNet-16":  {"층": 16,  "파라미터": 138_000_000, "특징": "3×3만 반복"},
    "GoogLeNet":  {"층": 22,  "파라미터":   7_000_000, "특징": "Inception 모듈"},
    "ResNet-152": {"층": 152, "파라미터":  25_000_000, "특징": "Skip Connection"},
}

print("┌──────────────┬──────┬──────────────┬──────────────────┐")
print("│ 구조         │ 층 수│ 파라미터 수  │ 핵심 특징        │")
print("├──────────────┼──────┼──────────────┼──────────────────┤")
for 이름, 정보 in 구조들.items():
    파라미터_만 = 정보["파라미터"] // 10000   # 만 단위로 변환
    print(f"│ {이름:<12s} │ {정보['층']:>4d} │ {파라미터_만:>8,d}만    │ {정보['특징']:<16s} │")
print("└──────────────┴──────┴──────────────┴──────────────────┘")

# VGG 대비 비율
vgg_파라미터 = 구조들["VGGNet-16"]["파라미터"]
print(f"\nVGG 대비 GoogLeNet 파라미터: {구조들['GoogLeNet']['파라미터'] / vgg_파라미터 * 100:.1f}% (약 1/20)")
print(f"VGG 대비 ResNet-152 파라미터: {구조들['ResNet-152']['파라미터'] / vgg_파라미터 * 100:.1f}% (약 1/5.5)")
print(f"→ ResNet은 층은 VGG의 약 10배인데, 파라미터는 오히려 적습니다!")

### Block 8 — 04-01-08-G 왜 GPU가 딥러닝에 잘 맞을까요? — 일개미 부대 비유

In [ ]:
import numpy as np

# ── 행렬곱에서 "동시에 계산 가능한 일감"이 얼마나 많은지 확인 ──
# 01-04 복습: 행렬곱은 "왼쪽의 가로줄 × 오른쪽의 세로줄을 곱해서 더하기"

# 예: 1000×1000 행렬 두 개를 곱하면?
행_수 = 1000
열_수 = 1000

# 결과 행렬의 각 칸은 서로 독립적으로 계산 가능합니다!
독립_계산_수 = 행_수 * 열_수
print(f"1000×1000 행렬곱에서 동시에 계산 가능한 일감: {독립_계산_수:,}개")

# ── CPU vs GPU 속도 비유 ──
cpu_코어 = 8        # 보통 CPU: 8코어 (슈퍼맨 8명)
gpu_코어 = 4096     # 보통 GPU: 4096코어 (일개미 4096마리)

# 각 코어가 한 번에 하나의 계산을 처리한다고 가정하면
cpu_반복 = 독립_계산_수 // cpu_코어
gpu_반복 = 독립_계산_수 // gpu_코어

print(f"\nCPU ({cpu_코어}코어): 한 코어가 {cpu_반복:,}번 일해야 함")
print(f"GPU ({gpu_코어}코어): 한 코어가 {gpu_반복:,}번 일해야 함")
print(f"\n→ GPU가 약 {cpu_반복 // gpu_반복}배 빠르게 끝낼 수 있습니다!")
print(f"→ 이것이 딥러닝에서 GPU를 사용하는 핵심 이유입니다.")

# ── 실제로 NumPy 행렬곱 실행해 보기 ──
A = np.random.rand(1000, 1000)
B = np.random.rand(1000, 1000)
C = A @ B   # @ = 행렬곱 (01-04 복습)
print(f"\n행렬곱 결과 크기: {C.shape} → 총 {C.shape[0] * C.shape[1]:,}개의 숫자를 계산했습니다")

### Block 9 — 📖 유도 — 결과부터 먼저, 그다음 한 줄씩

In [ ]:
import numpy as np

def softmax(a):
    c = np.max(a)              # 오버플로 방지: 최댓값을 빼줍니다
    exp_a = np.exp(a - c)     # 각 값에 지수함수 적용
    return exp_a / np.sum(exp_a)  # 전체 합으로 나누면 확률이 됩니다

def cross_entropy(y, t):
    delta = 1e-7               # log(0) 방지용 아주 작은 수
    return -np.sum(t * np.log(y + delta))

# 테스트 데이터
a = np.array([0.3, 2.9, 4.0])   # 신경망 출력값
t = np.array([0, 0, 1])           # 정답 (세 번째가 정답)
y = softmax(a)

# 수치미분으로 ∂L/∂a 직접 구하기
def L(a): return cross_entropy(softmax(a), t)

h = 1e-5
numeric_grad = np.zeros_like(a)
for i in range(len(a)):
    a_plus = a.copy(); a_plus[i] += h
    a_minus = a.copy(); a_minus[i] -= h
    numeric_grad[i] = (L(a_plus) - L(a_minus)) / (2 * h)

print("수치미분:", numeric_grad.round(4))
print("y − t:   ", (y - t).round(4))